# Design 7 — Sequential monitoring: boundaries, what they spend, and when they stop

A study that may stop on what it sees is not a study whose fixed-sample error rate
applies. `axiom.design.sequential` supplies three things and nothing else:

1. **boundaries** — thresholds on the interim statistic, one per look, from a classic
   shape (`pocock`, `obrien_fleming`), an alpha-spending function (`alpha_spending`), or
   a posterior-probability rule stated in words (`harm_boundary`);
2. **crossing probabilities** — `crossing_probabilities` / `operating_characteristics`
   integrate the canonical joint distribution exactly, so the error a rule *actually*
   spends is a computed number rather than a claimed one;
3. **the decision** — `monitor` walks realized statistics against a `StoppingRule` and
   returns the look it stopped at, plus a `LedgerLine` that names the bias of the
   estimate reported there.

The monitoring statistic is `Z_k = effect_k / se_k`, signed so **positive is better**,
and `t_k` is the information fraction. The B-values `B_k = Z_k · sqrt(t_k)` are a
Brownian motion with drift, and everything here is arithmetic on that one object.

In [ ]:
import numpy as np

from axiom.core import D, Unit
from axiom.design import (
    CANONICAL, STOPPED_ESTIMATE_BIAS, Boundary, BoundaryKind, CrossingProbabilities, Decision,
    LookOutcome, LookSchedule, MonitoringPath, OperatingCharacteristics, Side, SpendingFunction,
    StoppingRule, alpha_spending, crossing_probabilities, harm_boundary, information_fractions,
    monitor, obrien_fleming, operating_characteristics, pocock, sample_size, spending,
)

from axiom.display import enable, table

enable();  # every axiom result renders itself from here on

ALPHA, LOOKS = 0.05, 5
information = tuple((k + 1) / LOOKS for k in range(LOOKS))
print("information fractions:", information)
print("from cumulative enrollment:", information_fractions([48, 96, 144, 192, 240]))
print("...against a planned total of 300:", information_fractions([48, 96, 144, 192, 240], 300))

## 1. Three shapes for the same alpha

`pocock` judges every look at the same threshold; `obrien_fleming` uses `c / sqrt(t)`,
so the first look is nearly unstoppable. Both solve their one constant so the *total*
error under the null is exactly `alpha` — the reported constants are the published
ones (2.413 and 2.040 at five equally spaced looks, against the fixed-sample 1.960).
`alpha_spending` instead solves each threshold in turn against a budget, which is what
lets a committee move a look without invalidating the design.

In [ ]:
shapes = {
    "pocock": pocock(ALPHA, information),
    "obrien_fleming": obrien_fleming(ALPHA, information),
    "lan_demets_obf": alpha_spending(ALPHA, information, family="obrien_fleming"),
    "lan_demets_pocock": alpha_spending(ALPHA, information, family="pocock"),
}
table(
    [
        [name, " ".join(f"{v:.3f}" for v in boundary.z),
         f"{boundary.nominal_alpha(LOOKS - 1):.4f}",
         str([round(s, 4) for s in boundary.spent])]
        for name, boundary in shapes.items()
    ],
    headers=("shape", "z at each look", "final nominal p", "cumulative alpha spent"),
)

The spending functions themselves, evaluated anywhere in `[0, 1]`, are what the
thresholds are solved against. `"power"` with `rho` is the tunable family between them.

In [ ]:
grid = (0.1, 0.25, 0.5, 0.75, 1.0)
rows = []
for kind in ("obrien_fleming", "pocock", "power"):
    kind_: SpendingFunction = kind
    rows.append([kind, *[f"{spending(kind_, t, ALPHA):.4f}" for t in grid]])
rows.append(["power (rho=3)", *[f"{spending('power', t, ALPHA, rho=3.0):.4f}" for t in grid]])
table(rows, headers=("spending function", *[f"t={t}" for t in grid]))

## 2. What a shape costs

Every look bought is paid for at the final analysis. Pin the drift the design is
powered for — for a fixed-sample two-arm test at 80 % power and two-sided 5 %, the
expected Z at full information is 2.802 — and read `operating_characteristics` for
the power each shape actually delivers and the information it expects to use.

In [ ]:
ss = sample_size(effect=8.0, sd=14.0, power=0.8)
drift = 8.0 / (14.0 * np.sqrt(4.0 / ss.n))
print(f"fixed-sample n = {ss.n} at 80% power; drift at full information = {drift:.3f}")

schedule = LookSchedule(labels=tuple(f"look_{k + 1}" for k in range(LOOKS)), information=information)
rows = []
for name, boundary in shapes.items():
    rule = StoppingRule(name=name, looks=schedule, boundaries=(boundary,))
    null = operating_characteristics(rule, 0.0)
    alt = operating_characteristics(rule, drift)
    rows.append(
        [name, f"{null.crossings.cumulative('efficacy'):.4f}",
         f"{alt.crossings.cumulative('efficacy'):.3f}",
         f"{alt.expected_information:.3f}", f"{alt.expected_looks:.2f}"]
    )
table(rows, headers=("shape", "type I", "power", "E[information]", "E[looks]"))

Pocock stops soonest — the smallest expected information — and pays for it with the
lowest power, because its final critical value is the largest. O'Brien–Fleming keeps
almost all the fixed-sample power and still stops early when the effect is large.

## 3. A harm boundary is a sentence, not a z

`harm_boundary` takes the rule a monitoring committee writes down — *stop when the
posterior probability that the treatment is worse than the control by more than
`margin` reaches `probability`* — and returns the Z threshold that implements it,
look by look. With `margin = 0` the threshold is constant; a positive margin makes the
early looks *harder* to cross, because a large observed harm is cheap to come by when
the standard error is large.

In [ ]:
se_final = 14.0 * float(np.sqrt(4.0 / ss.n))
flat = harm_boundary(0.95, information)
with_margin = harm_boundary(0.95, information, margin=3.0, se_at_full_information=se_final)
print("se at full information:", round(se_final, 3))
print("margin 0 :", [round(v, 3) for v in flat.z])
print("margin 3 :", [round(v, 3) for v in with_margin.z])
print("detail   :", with_margin.detail)
kind: BoundaryKind = with_margin.kind
side: Side = with_margin.side
print(f"kind={kind} side={side} binding={with_margin.binding}")

A probability rule states a posterior, not an error rate. `crossing_probabilities`
says what it spends: this one stops a *null* study for harm about one time in thirty.

In [ ]:
harm_only = StoppingRule(name="harm_only", looks=schedule, boundaries=(with_margin,))
spent = crossing_probabilities(harm_only, 0.0)
assert isinstance(spent, CrossingProbabilities)
print("P(stop for harm | no true difference) =", round(spent.cumulative("harm"), 4))
print("per look:", [round(p, 4) for p in spent.per_look["harm"]])
print("P(never stop) =", round(spent.continue_probability, 4))

## 4. A rule is not the sum of its boundaries

`StoppingRule` composes an efficacy boundary, a harm boundary and a non-binding
futility boundary into one continuation region per look. Two effects follow, and both
are computed rather than asserted: the efficacy boundary loses some of its own alpha to
paths another boundary stops first, and *ignoring* the non-binding futility boundary —
the conservative convention — overstates the error the rule as run spends. Here the
harm boundary costs the efficacy boundary almost nothing, because the two sit on
opposite sides of zero and a path that reaches one rarely reaches the other; futility
costs it more. Neither is a fact about boundaries in general, which is the point of
computing it.

In [ ]:
efficacy = alpha_spending(ALPHA / 2, information, family="obrien_fleming", side="upper")
futility = Boundary(kind="futility", side="lower", z=(-1.5, -0.8, -0.2, 0.3, 0.8), binding=False)
rule = StoppingRule(name="hyper3", looks=schedule, boundaries=(efficacy, with_margin, futility))
rows = []
for look in range(LOOKS):
    low, high = rule.continuation(look)
    rows.append([look + 1, f"{low:+.3f}", f"{high:+.3f}"])
table(rows, headers=("look", "continue while Z >", "and Z <"))

alone = crossing_probabilities(
    StoppingRule(name="e", looks=schedule, boundaries=(efficacy,)), 0.0
).cumulative("efficacy")
binding = crossing_probabilities(rule, 0.0)
as_run = crossing_probabilities(rule, 0.0, binding_only=False)
print(f"\n{'efficacy alone':22s} {alone:.6f}")
print(f"{'with harm added':22s} {binding.cumulative('efficacy'):.6f}"
      f"   (harm itself takes {binding.cumulative('harm'):.4f})")
print(f"{'as run, futility on':22s} {as_run.cumulative('efficacy'):.6f}"
      f"   (futility takes {as_run.cumulative('futility'):.4f})")
print("binding_only:", binding.binding_only, as_run.binding_only)

## 5. Operating characteristics across the truth

`operating_characteristics` at a grid of drifts is the design's whole story: how often
it stops, for what reason, and how much of the planned enrollment it expects to use.
Note the asymmetry the design was built for — a harmful treatment is stopped far
earlier than a beneficial one is confirmed.

In [ ]:
print(f"{'drift':>7} {'efficacy':>9} {'harm':>7} {'futility':>9} {'E[info]':>8} {'E[looks]':>9}")
rows = []
for d in (-3.5, -2.0, -1.0, 0.0, 1.0, 2.0, 2.802, 3.5):
    oc = operating_characteristics(rule, d, binding_only=False)
    assert isinstance(oc, OperatingCharacteristics)
    c = oc.crossings
    rows.append(
        [f"{d:.2f}", f"{c.cumulative('efficacy'):.3f}", f"{c.cumulative('harm'):.3f}",
         f"{c.cumulative('futility'):.3f}", f"{oc.expected_information:.3f}",
         f"{oc.expected_looks:.2f}"]
    )
table(rows, headers=("drift", "efficacy", "harm", "futility", "E[information]", "E[looks]"))
harmful = operating_characteristics(rule, -2.802, binding_only=False)
print("\nstop probability at drift -2.802:", round(harmful.stop_probability, 4))
print("stops by look:", [round(p, 3) for p in harmful.crossings.by_look()])

## 6. Running one study against the rule

`monitor` takes the realized `Z` at each look taken so far. It stops at the first
crossing, and the outermost boundary wins when two fire at once — a statistic below
the harm boundary is a harm stop even though it is also below futility.

In [ ]:
observed_z = [-0.42, -0.55, -3.05, 0.0, 0.0]
effects = [-1.8, -2.0, -9.6, 0.0, 0.0]
ses = [4.3, 3.7, 3.1, 2.7, 2.4]
path = monitor(rule, observed_z[:3], effects=effects[:3], ses=ses[:3])
assert isinstance(path, MonitoringPath)
rows = []
for outcome in path.looks:
    assert isinstance(outcome, LookOutcome)
    low, high = rule.continuation(outcome.look)
    rows.append(
        [outcome.label, f"{outcome.information:.2f}", f"{outcome.z:+.3f}",
         f"({low:+.2f}, {high:+.2f})", outcome.decision]
    )
table(rows, headers=("look", "information", "Z", "continuation", "decision"))
decision: Decision = path.decision
print(f"\ndecision={decision} at look {path.stopped_at + 1}, threshold {path.threshold():.3f}")
print("crossed the boundary:", rule.of_kind("harm") is rule.crossings(2, -3.05)[0])

The number that leaves the study carries the reason it is suspect. `monitor` emits a
`LedgerLine` whose assumption is `stopped_estimate_bias` when a boundary stopped it and
`canonical_joint_distribution` when it did not — rule 4 of the repo, applied to the one
estimate a sequential design is most likely to over-read.

In [ ]:
line = path.ledger_line()
print(line.kind, "|", line.statement)
print("assumption:", line.assumption.name, "-", line.assumption.statement)
print("challenged by:", line.assumption.challenged_by)
print("detail:", line.detail)
print("\nstill running:", monitor(rule, [0.1, 0.4]).ledger_line().assumption.name)
print("named assumptions:", CANONICAL.name, "|", STOPPED_ESTIMATE_BIAS.name)

## 7. Looks that did not happen when they were planned

The point of a spending function: recompute the remaining thresholds against the
information that actually accrued. Enrollment ran slow, so the third look happens at
52 % rather than 60 % — only the thresholds from there on move.

In [ ]:
planned = information
actual = (0.2, 0.4, 0.52, 0.79, 1.0)
rows = []
for label, fractions in (("planned", planned), ("actual", actual)):
    b = alpha_spending(ALPHA / 2, fractions, family="obrien_fleming", side="upper")
    rows.append(
        [label, str([round(t, 2) for t in fractions]), str([round(v, 3) for v in b.z]),
         f"{b.spent[-1]:.4f}"]
    )
table(rows, headers=("schedule", "information fractions", "z", "total spent"))

## 8. The dimension the statistic is not carrying

`Z` is dimensionless by construction — an effect divided by its standard error — which
is why one module serves an outcome measured in mmHg and one measured in anything else.
The units live with the effect, not with the boundary.

In [ ]:
mmhg = Unit(name="mmHg", dimension=D.outcome)
print("outcome unit:", mmhg.name, "| dimension:", mmhg.dimension)
print("effect at the stopping look:", path.looks[-1].effect, mmhg.name)
print("its se:", path.looks[-1].se, mmhg.name, "-> Z =", round(path.looks[-1].z, 3), "(dimensionless)")

## What this notebook decided

- The shape is a trade, and it is priced: Pocock stops soonest and has the least power,
  O'Brien–Fleming keeps nearly the fixed-sample power and still stops early when the
  effect is large. `operating_characteristics` gives both numbers for both.
- A posterior-probability harm rule is a legitimate boundary, and
  `crossing_probabilities` says what it spends. Stating a posterior is not the same as
  controlling an error rate, and the design should know both numbers.
- Boundaries interact. Adding a harm boundary takes paths away from the efficacy
  boundary; treating futility as non-binding overstates what the rule as run spends.
  `binding_only` makes the gap a number.
- An estimate reported at the look that stopped the study is biased away from the null,
  and the ledger line says so rather than letting the number travel alone.